# CAUCHY Phase 7.0 — RS Field Generation on Binder
Genera campi 3D in redshift space (128³, PCS) da snapshot Quijote nwLH.
Output: `df_m_128_RS_z=0.npy` per ogni sim — scaricabili via Jupyter file browser.

**Parametri frozen:** σ_px=0.640, MAS=PCS, LOS=z-axis, H(z=0)=67.11 km/s/(Mpc/h)

In [ ]:
# ── Cell 1: configurazione ────────────────────────────────────────────────────
# Adatta SNAP_ROOT al path reale su Binder (esegui Cell 2 prima per trovarlo)
SNAP_ROOT   = '/home/jovyan/Snapshots/latin_hypercube_nwLH'  # aggiusta se necessario
OUTPUT_DIR  = '/home/jovyan/RS_fields'   # cartella output su Binder

SNAP_SUBDIR = 'snapdir_004'   # z=0
SNAP_PREFIX = 'snap_004'

# Parametri cosmologici fissi nwLH
BOXSIZE   = 1000.0   # Mpc/h
HUBBLE_Z0 = 67.11    # H(z=0) in km/s/(Mpc/h)
REDSHIFT  = 0.0
LOS_AXIS  = 2        # piano-parallelo, LOS lungo z
GRID      = 128
MAS       = 'PCS'

# Batch da processare in questa sessione — modifica per ogni sessione Binder
# Sessione 1: range(0, 200, 10)   → sim 0,10,...,190   (20 sim)
# Sessione 2: range(200, 400, 10) → sim 200,210,...,390 (20 sim)
# ... etc fino a range(1800, 2000, 10)
BATCH_INDICES = list(range(0, 200, 10))   # <-- modifica per ogni sessione

print(f'Batch: {len(BATCH_INDICES)} sim, indici {BATCH_INDICES[0]}..{BATCH_INDICES[-1]}')

In [ ]:
# ── Cell 2: trova il path corretto degli snapshot su Binder ──────────────────
import os

# Possibili path su Binder/cluster Quijote
candidates = [
    '/home/jovyan/Snapshots/latin_hypercube_nwLH',
    '/mnt/ceph/users/fvillaescusa/Quijote/Snapshots/latin_hypercube_nwLH',
    '/simons/scratch/fvillaescusa/quijote/Snapshots/latin_hypercube_nwLH',
    '/projects/QUIJOTE/Snapshots/latin_hypercube_nwLH',
]

for p in candidates:
    exists = os.path.exists(p)
    # try first sim
    snap0 = os.path.join(p, '0', 'snapdir_004', 'snap_004.0.hdf5')
    snap0_exists = os.path.exists(snap0)
    print(f'{"OK" if exists else "--"} dir:  {p}')
    if exists:
        print(f'   snap_004.0.hdf5 present: {snap0_exists}')
        if snap0_exists:
            print(f'   >>> USE THIS PATH')

In [ ]:
# ── Cell 3: imports e verifica Pylians ────────────────────────────────────────
import numpy as np
import h5py
import hdf5plugin  # per snapshot compressi Blosc
import MAS_library as MASL
import redshift_space_library as RSL
from pathlib import Path

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print('Imports OK')
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
# ── Cell 4: funzione di generazione campo RS ──────────────────────────────────
def make_rs_field(idx):
    """
    Legge snapshot HDF5 nwLH sim <idx>, applica RSD, voxelizza 128^3 PCS.
    Restituisce (delta_float32, status_str).
    """
    snap_dir = os.path.join(SNAP_ROOT, str(idx), SNAP_SUBDIR)
    if not os.path.exists(snap_dir):
        return None, 'snap_dir_not_found'

    # Trova tutti i subfile HDF5
    import glob
    files = sorted(glob.glob(os.path.join(snap_dir, SNAP_PREFIX + '*.hdf5')))
    if not files:
        return None, 'no_hdf5_files'

    pos_list, vel_list = [], []
    for fpath in files:
        with h5py.File(fpath, 'r') as f:
            # PartType1 = dark matter
            pos = f['PartType1/Coordinates'][:].astype(np.float32)
            vel = f['PartType1/Velocities'][:].astype(np.float32)
            # Gadget internal units: positions in kpc/h → Mpc/h
            pos /= 1000.0
            # Velocities: at z=0, a=1, v_gadget = v_pec (km/s) already
            pos_list.append(pos)
            vel_list.append(vel)

    pos = np.concatenate(pos_list, axis=0)
    vel = np.concatenate(vel_list, axis=0)

    # Applica RSD piano-parallelo lungo LOS_AXIS
    RSL.pos_redshift_space(pos, vel, BOXSIZE, HUBBLE_Z0, REDSHIFT, LOS_AXIS)

    # Voxelizza su griglia 128^3 con MAS PCS
    delta = np.zeros((GRID, GRID, GRID), dtype=np.float32)
    MASL.MA(pos, delta, BOXSIZE, MAS)

    # Densità contrasto: delta = n/<n> - 1
    mean_n = np.mean(delta, dtype=np.float64)
    if mean_n == 0:
        return None, 'zero_mean'
    delta = delta.astype(np.float64)
    delta /= mean_n
    delta -= 1.0

    return delta.astype(np.float32), 'ok'

print('Funzione make_rs_field definita.')

In [ ]:
# ── Cell 5: test su 1 sim prima del batch completo ────────────────────────────
test_idx = BATCH_INDICES[0]
print(f'Test su sim {test_idx}...')
delta, status = make_rs_field(test_idx)
if delta is None:
    print(f'ERRORE: {status}')
else:
    print(f'OK — shape: {delta.shape}, dtype: {delta.dtype}')
    print(f'     mean={delta.mean():.6f}, std={delta.std():.4f}, min={delta.min():.4f}, max={delta.max():.4f}')
    # Salva test
    out = os.path.join(OUTPUT_DIR, f'df_m_128_RS_z=0_sim{test_idx}.npy')
    np.save(out, delta)
    size_mb = os.path.getsize(out) / 1e6
    print(f'     Salvato: {out} ({size_mb:.1f} MB)')

In [ ]:
# ── Cell 6: batch completo ────────────────────────────────────────────────────
# Esegui solo dopo che Cell 5 ha dato OK
import json, time

results = {'ok': [], 'error': {}}
t0 = time.time()

for i, idx in enumerate(BATCH_INDICES):
    out_path = os.path.join(OUTPUT_DIR, f'df_m_128_RS_z=0_sim{idx}.npy')

    if os.path.exists(out_path):
        results['ok'].append(idx)
        continue

    delta, status = make_rs_field(idx)
    if delta is None:
        results['error'][str(idx)] = status
        print(f'[{i+1}/{len(BATCH_INDICES)}] sim {idx}: ERRORE ({status})')
    else:
        np.save(out_path, delta)
        results['ok'].append(idx)
        elapsed = time.time() - t0
        eta = elapsed / (i+1) * (len(BATCH_INDICES) - i - 1)
        print(f'[{i+1}/{len(BATCH_INDICES)}] sim {idx}: OK — '
              f'mean={delta.mean():.4f}  ETA: {eta/60:.1f} min')

print(f'\nBatch completato: {len(results["ok"])} OK, {len(results["error"])} errori')
print(f'File in: {OUTPUT_DIR}')
print(json.dumps(results, indent=2))

In [ ]:
# ── Cell 7: crea archivio .tar per download ───────────────────────────────────
# Crea un tar con tutti i file del batch corrente per scaricarli in un colpo
import tarfile

batch_start = BATCH_INDICES[0]
batch_end   = BATCH_INDICES[-1]
tar_name    = f'/home/jovyan/RS_fields_batch_{batch_start}_{batch_end}.tar'

with tarfile.open(tar_name, 'w') as tar:
    for idx in results['ok']:
        f = os.path.join(OUTPUT_DIR, f'df_m_128_RS_z=0_sim{idx}.npy')
        if os.path.exists(f):
            tar.add(f, arcname=f'df_m_128_RS_z=0_sim{idx}.npy')

size_mb = os.path.getsize(tar_name) / 1e6
print(f'Archivio creato: {tar_name} ({size_mb:.0f} MB)')
print('Scaricalo dal file browser di Jupyter (pannello sinistro)')